In [1]:
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import math

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from tqdm import tqdm


device = torch.device("cuda" if torch.cuda.is_available()else "cpu")

In [2]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

train_df_small = train_df.iloc[:10000]
val_df_small = val_df.iloc[:2000]
test_df_small = test_df.iloc[:2000]

def tokenizer(code):
    return code.split()

In [3]:
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

counter = Counter()

for _, row in train_df_small.iterrows():
    counter.update(tokenizer(row["buggy"]))
    counter.update(tokenizer(row["fixed"]))

vocab = {
    PAD: 0,
    SOS: 1,
    EOS: 2,
    UNK: 3
}

for token, _ in counter.items():
    vocab[token] = len(vocab)

words = {idx: word for word, idx in vocab.items()}

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 415


In [4]:
def encode(tokens):
    ids = [vocab[SOS]]
    for token in tokens:
        ids.append(vocab.get(token, vocab[UNK]))
    ids.append(vocab[EOS])
    return torch.tensor(ids)

In [5]:
class CodeFix(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        source = encode(tokenizer(row["buggy"]))
        target = encode(tokenizer(row["fixed"]))

        return source, target

In [6]:
PAD_IDX = vocab[PAD]

def padder(batch):
    src_batch = [item[0] for item in batch]
    trg_batch = [item[1] for item in batch]

    src_batch = pad_sequence(src_batch,padding_value=PAD_IDX)
    trg_batch = pad_sequence(trg_batch,padding_value=PAD_IDX)

    return src_batch, trg_batch

In [7]:
train_dataset = CodeFix(train_df_small)
val_dataset = CodeFix(val_df_small)
test_dataset = CodeFix(test_df_small)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,collate_fn=padder)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False,collate_fn=padder)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,collate_fn=padder)

In [8]:
def train(model,loader,optimizer,criterion):
    model.train()
    epoch_loss = 0
    for src, trg in tqdm(loader):
        src = src.to(device)
        trg = trg.to(device)
        optimizer.zero_grad()
        output = model(src,trg[:-1])
        output = output.reshape(-1,output.shape[-1])
        trg = trg[1:].reshape(-1)
        loss = criterion(output,trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [9]:
def validate(model,loader,criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in loader:
            src = src.to(device)
            trg = trg.to(device)
            output = model(src,trg[:-1])
            output = output.reshape(-1,output.shape[-1])
            trg = trg[1:].reshape(-1)
            loss = criterion(output,trg)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [10]:
def repair_transformer(code, model, max_len=100):
    model.eval()
    tokens = tokenizer(code)
    src = encode(tokens).unsqueeze(1).to(device)
    generated = [vocab[SOS]]
    with torch.no_grad():
        for _ in range(max_len):
            trg = torch.tensor(generated,dtype=torch.long,device=device).unsqueeze(1)
            output = model(src, trg)
            next_token = output[-1].argmax(1).item()
            if next_token == vocab[EOS]:
                break
            generated.append(next_token)
    return " ".join(words[idx]for idx in generated[1:])

In [11]:
def run(model,train_loader,val_loader,epochs=5,lr=0.001,model_name="Model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = optim.Adam(model.parameters(),lr=lr)
    train_losses = []
    val_losses = []
    for epoch in range(epochs):
        train_loss = train(model,train_loader,optimizer,criterion)
        val_loss = validate(model,val_loader,criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"{model_name} Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    return model, train_losses, val_losses

In [12]:
def evaluate_transformer(model, dataframe, samples=1000):
    model.eval()
    correct_tokens = 0
    total_tokens = 0
    for i in tqdm(range(min(samples, len(dataframe))),desc="Evaluating"):
        row = dataframe.iloc[i]
        pred = repair_transformer(row["buggy"],model).split()
        actual = row["fixed"].split()
        m = min(len(pred),len(actual))
        for j in range(m):
            if pred[j] == actual[j]:
                correct_tokens += 1
        total_tokens += len(actual)
    return correct_tokens / total_tokens

In [13]:
class Positional(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0,max_len,dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer("pe",pe)
    def forward(self, x):
        return (x+self.pe[:x.size(0)])

In [14]:
class SelfAttention(nn.Module):

    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value, mask=None):
        Q = self.Wq(query)
        K = self.Wk(key)
        V = self.Wv(value)
        Q = Q.permute(1,0,2)
        K = K.permute(1,0,2)
        V = V.permute(1,0,2)
        scores = torch.matmul(Q,K.transpose(-2, -1)) / math.sqrt(self.embed_dim)
        if mask is not None:scores = scores.masked_fill(mask == 0,float("-inf"))
        attention = torch.softmax(scores,dim=-1)
        output = torch.matmul(attention,V)
        output = output.permute(1,0,2)
        return output

In [15]:
class HeadAttention(nn.Module):
    def __init__(self, embed_dim, nhead):
        super().__init__()
        self.head_dim = embed_dim//nhead
        self.nhead = nhead
        self.heads = nn.ModuleList([SelfAttention(self.head_dim) for _ in range(nhead)])
        self.fc = nn.Linear(embed_dim,embed_dim)
        
    def forward(self,query,key,value,mask=None):
        batch_size = query.shape[1]
        outputs = []
        for i, head in enumerate(self.heads):
            start = i * self.head_dim
            end = start + self.head_dim
            q = query[:, :, start:end]
            k = key[:, :, start:end]
            v = value[:, :, start:end]
            outputs.append(head(q,k,v,mask))
        output = torch.cat(outputs,dim=-1)
        return self.fc(output)

In [16]:
class Forward(nn.Module):
    def __init__(self,embed_dim,ff_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

In [17]:
class Encoder(nn.Module):
    def __init__(self,embed_dim,nhead,ff_dim,dropout):
        super().__init__()
        self.attn = HeadAttention(embed_dim,nhead)
        self.ff = Forward(embed_dim,ff_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        attn = self.attn(src,src,src)
        src = self.norm1(src +self.dropout(attn))
        ff = self.ff(src)
        src = self.norm2(src +self.dropout(ff))
        return src

In [18]:
class Decoder(nn.Module):
    def __init__(self,embed_dim,nhead,ff_dim,dropout):
        super().__init__()
        self.self_attn = HeadAttention(embed_dim,nhead)
        self.cross_attn = HeadAttention(embed_dim,nhead)
        self.ff = Forward(embed_dim,ff_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self,trg,memory,mask):
        attn = self.self_attn(trg,trg,trg,mask)
        trg = self.norm1(trg +self.dropout(attn))
        attn = self.cross_attn(trg,memory,memory)
        trg = self.norm2(trg +self.dropout(attn))
        ff = self.ff(trg)
        trg = self.norm3(trg +self.dropout(ff))
        return trg

In [19]:
class Transformer(nn.Module):
    def __init__(self,vocab_size,embed_dim,nhead,num_layers,ff_dim,dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=PAD_IDX)
        self.pos_encoder = Positional(embed_dim)
        self.encoder_layers = nn.ModuleList([Encoder(embed_dim,nhead,ff_dim,dropout)for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([Decoder(embed_dim,nhead,ff_dim,dropout)for _ in range(num_layers)])
        self.fc = nn.Linear(embed_dim,vocab_size)
   
    def make_mask(self,trg_len,device):
        mask = torch.tril(torch.ones(trg_len,trg_len,device=device))
        return mask

    def forward(self,src,trg):
        src = self.embedding(src)
        trg = self.embedding(trg)
        src = self.pos_encoder(src)
        trg = self.pos_encoder(trg)
        memory = src
        for layer in self.encoder_layers:
            memory = layer(memory)
        mask = self.make_mask(trg.size(0),trg.device)
        output = trg
        for layer in self.decoder_layers:
            output = layer(output,memory,mask)
        return self.fc(output)

In [20]:
VOCAB_SIZE = len(vocab)
EMBED_DIM = 128
NHEAD = 8
NUM_LAYERS = 3
FF_DIM = 512

In [22]:
transformer = Transformer(VOCAB_SIZE,EMBED_DIM,NHEAD,NUM_LAYERS,FF_DIM).to(device)
transformer, train_hist, val_hist = run(transformer,train_loader,val_loader,epochs=5,model_name="Transformer")
transformer_acc = evaluate_transformer(transformer,val_df,1000)
print("Transformer Token Accuracy:",transformer_acc * 100)

torch.save(transformer.state_dict(),"transformer.pth")

100%|██████████| 313/313 [00:58<00:00,  5.31it/s]


Transformer Epoch 1/5 | Train Loss: 1.9020 | Val Loss: 1.3645


100%|██████████| 313/313 [00:42<00:00,  7.42it/s]


Transformer Epoch 2/5 | Train Loss: 1.2400 | Val Loss: 0.9926


100%|██████████| 313/313 [00:45<00:00,  6.88it/s]


...

Transformer Epoch 4/5 | Train Loss: 0.8289 | Val Loss: 0.7026


100%|██████████| 313/313 [00:52<00:00,  5.93it/s]


Transformer Epoch 5/5 | Train Loss: 0.7395 | Val Loss: 0.6417


Evaluating: 100%|██████████| 1000/1000 [26:07<00:00,  1.57s/it] 

Transformer Token Accuracy: 48.60423542354235


In [24]:
torch.save(vocab, "transformer_vocab.pt")
torch.save(words, "transformer_words.pt")